# Figure 6.3: homogeneous versus heterogeneous ages

Compare the dynamic hedging-cost distributions of **1,000 lives aged 60**
with **1,000 lives in ten age groups: 55.5, 56.5, ..., 64.5**, separately
for PE, TI and EI. Each heterogeneous group contains 100 lives. This is the
age allocation in `reference/hedge_cost.cpp`; the paper describes its range
as 55-65. Both populations have mean age 60 and the same total notional.
Changing age changes mortality, so each population gets its **own computed
initial premium and deltas**.

Every model and numerical input is written in the cells below. No demo or
paper preset is loaded. Costs follow the current library convention: present
values at time zero, positive for additional funding. Dynamic hedging is
implemented on the chosen discrete outer grid. The published legacy cost
values are not exact acceptance targets for the corrected accounting.

**Workspace and execution controls.** Keep this notebook in the `notebook/` folder
or its `notebooks/` directory and select the `.venv-analysis` Python kernel.
The setup cell detects the repository root from the working directory. The C++ executable must support OpenMP for
multiple outer threads. Build it in a terminal:

```bash
# Run from the repository root.
cmake -S . -B build-paper-release -DBoost_ROOT=/usr \
    -DCMAKE_BUILD_TYPE=Release -DCARBONFUND_ENABLE_OPENMP=ON -DBUILD_TESTING=ON
cmake --build build-paper-release --target carbonfund_experiment --parallel
```

`RUN_SIMULATIONS=True` runs the six default experiments.
`REUSE_COMPLETED_RUNS=True` reuses completed runs only when their simulation
inputs match exactly; this also lets you resume between completed experiments.
`RUN_SIMULATIONS=False` only reloads existing CSVs and recomputes analyses.
`OVERWRITE_EXISTING=True` explicitly reruns experiments in their old directories.
Prefer a new `RUN_NAME` after changing simulation inputs. Analysis and plotting
inputs may change without rerunning C++. Raw CSVs are written after each
experiment finishes, not as an in-progress scenario checkpoint.

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import math
import sys

SEARCH_START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (parent for parent in (SEARCH_START, *SEARCH_START.parents)
     if (parent / 'include' / 'carbonfund').is_dir()
     and (parent / 'python' / 'carbonfund_experiments.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start the notebook from the CarbonFund_UL folder or a subdirectory.')
BINARY = PROJECT_ROOT / 'build-paper-release' / ('carbonfund_experiment.exe' if sys.platform == 'win32' else 'carbonfund_experiment')
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'homogeneous_vs_heterogeneous'
RUN_NAME = 'cir_T20_figure_6_3_01'
RUN_SIMULATIONS = True
REUSE_COMPLETED_RUNS = True
OVERWRITE_EXISTING = False
SAVE_FIGURES = True
SHOW_FIGURES = True
PRINT_PROGRESS = True

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Edit PROJECT_ROOT first: {PROJECT_ROOT}')
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME must be a single directory name.')
sys.path.insert(0, str(PROJECT_ROOT / 'python'))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from scipy.stats import gaussian_kde, t as student_t
from IPython.display import HTML, display
import carbonfund_experiments as cfe

RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME
print('Python:', sys.executable)
print('Python interface:', Path(cfe.__file__).resolve())
print('C++ executable:', BINARY)
print('Output directory:', RUN_DIRECTORY)


**Market inputs.** There are exactly four stocks. Returns and volatilities are
annualized; the risk-free rate is continuously compounded. The correlation
matrix must be symmetric and positive definite with unit diagonal. The code
constructs covariance from volatilities and correlations. These are the
full-precision legacy correlations; Table 4.1 prints rounded entries.

In [ ]:
market = {
    'risk_free_rate': 0.05,
    'expected_returns': [0.25, 0.15, 0.10, 0.08],
    'volatilities': [0.30, 0.25, 0.20, 0.16],
    'correlations': [
        [1.0000, 0.4397, 0.3900, 0.3168],
        [0.4397, 1.0000, 0.2954, 0.3261],
        [0.3900, 0.2954, 1.0000, 0.3134],
        [0.3168, 0.3261, 0.3134, 1.0000],
    ],
}


**Carbon inputs.** The paper uses four CIR intensities,
$dC_i=\kappa_i(\theta_i-C_i)dt+\sigma_i\sqrt{C_i}\,dB_i$,
simulated by the library's CIR scheme. All four entries can be changed.
The other supported model names are `OU` and `GeoOU`.
OU uses the same parameter keys with additive noise. GeoOU models the log
intensity as OU; replace `long_run_mean` with `long_run_log_mean` when selecting
it, and keep initial intensities positive. For CIR, initial and long-run
intensities are nonnegative. Stocks share the chosen market correlations;
the current carbon components and financial innovations are independent.

In [ ]:
carbon = {
    'model': 'CIR',
    'initial': [5000.0, 4000.0, 3000.0, 1000.0],
    'long_run_mean': [2500.0, 2000.0, 1500.0, 500.0],
    'mean_reversion': [0.05, 0.05, 0.05, 0.05],
    'volatility': [3.0, 3.0, 3.0, 3.0],
}


**Fund inputs.** Risk aversion and carbon penalization determine the fund's
stock weights from the observed carbon state. Larger carbon penalization
discourages exposure to carbon-intensive stocks. These preferences and the
market/carbon inputs remain common to all portfolio sizes and contract types.

In [ ]:
fund = {
    'risk_aversion': 1.0,
    'carbon_penalty': 0.0025,
}


**Contract inputs.** The benefit at elapsed time $t$ is
$\min(U_0e^{g_Ut},\max(L_0e^{g_Lt},X_t))$ per unit of notional.
PE pays on survival at maturity; TI pays on death before maturity; EI combines
survival payment and a death payment multiplied by the death-benefit fraction.
These are nominal benefits, discounted once in present-value costs.
Guarantees stay anchored to these original terms when the observed fund changes.
`fund_value` is the initial unit fund value, not the portfolio's total notional.
The growth rates are explicit inputs; changing the market rate does not
automatically change them.

In [ ]:
contract = {
    'fund_value': 1.0,
    'maturity': 20.0,
    'minimum_guarantee': 1.0,
    'maximum_guarantee': 1.0,
    'minimum_guarantee_growth_rate': 0.05,
    'maximum_guarantee_growth_rate': 0.50,
    'endowment_death_benefit_fraction': 1.0,
}


**Mortality inputs.** For a policyholder aged $a$ initially,
$\mu(a+t)=\xi+b^{-1}\exp((a+t-m)/b)$.
This uses the corrected minus sign and $\xi=0.0041959$.
The scale is `b`, not its reciprocal. Lifetimes are independent across
policyholders under this deterministic mortality law; there is no common
stochastic mortality factor in this experiment.

In [ ]:
mortality = {
    'xi': 0.0041959,
    'm': 79.6921211,
    'b': 11.5818911,
}


**Population inputs.** Every row is an age cohort; `count` is its number of
independent insured lives and `notional_per_policy` is each person's exposure.
The notebook expands these rows into the individual-policy input expected by
C++. Edit every age, count and notional directly here. Cohorts stay in the
listed order, which matters for reproducible mortality draws.

The default per-policy notional is 0.001 in **both** populations, giving total
notional 1 each. This follows our other notebooks; `per_notional` analysis
also makes comparisons independent of a common exposure scale. To use the
legacy unit notional, change every `notional_per_policy` to 1.0; aggregate
costs then scale by 1,000. Population sizes and total notionals must match
between the two populations to isolate age composition.

The ten midpoint ages and their counts come from the legacy code; the current
simulation budgets and seeds below are explicit choices. There is no random
sampling of initial ages. Expected survivors are a deterministic mortality
diagnostic, not a replacement for simulating individual deaths.

In [ ]:
CONTRACT_TYPES = ['PE', 'TI', 'EI']

HOMOGENEOUS_GROUPS = [
    {'age': 60.0, 'count': 1000, 'notional_per_policy': 0.001},
]
HETEROGENEOUS_GROUPS = [
    {'age': 55.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 56.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 57.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 58.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 59.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 60.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 61.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 62.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 63.5, 'count': 100, 'notional_per_policy': 0.001},
    {'age': 64.5, 'count': 100, 'notional_per_policy': 0.001},
]

POPULATION_ORDER = ('homogeneous', 'heterogeneous')
population_groups = dict(zip(POPULATION_ORDER, (HOMOGENEOUS_GROUPS, HETEROGENEOUS_GROUPS)))
population_labels = {'homogeneous': 'Homogeneous', 'heterogeneous': 'Heterogeneous'}
if (not CONTRACT_TYPES or len(set(CONTRACT_TYPES)) != len(CONTRACT_TYPES)
        or any(k not in ('PE', 'TI', 'EI') for k in CONTRACT_TYPES)):
    raise ValueError('Select distinct PE, TI and/or EI contract types.')

profiles = []
population_totals = {}
for population_id, groups in population_groups.items():
    if not isinstance(groups, list) or not groups:
        raise ValueError('Each population needs a nonempty list of groups.')
    for group in groups:
        if set(group) != {'age', 'count', 'notional_per_policy'}:
            raise ValueError('Each group needs age, count and notional_per_policy.')
        age, count, q = group['age'], group['count'], group['notional_per_policy']
        if isinstance(count, bool) or not isinstance(count, int) or count < 0:
            raise ValueError('Group counts must be nonnegative integers.')
        if any(isinstance(v, bool) or not isinstance(v, (int, float))
               or not math.isfinite(v) or v < 0 for v in (age, q)):
            raise ValueError('Ages and notionals must be finite and nonnegative.')
        integrated_hazard = (mortality['xi'] * contract['maturity']
            + math.exp((age - mortality['m']) / mortality['b'])
            * math.expm1(contract['maturity'] / mortality['b']))
        survival = math.exp(-integrated_hazard)
        profiles.append({
            'population': population_id, **group,
            'total_notional': count * q,
            'survival_probability_at_maturity': survival,
            'expected_survivors_at_maturity': count * survival,
        })
    count = sum(g['count'] for g in groups)
    total = math.fsum(g['count'] * g['notional_per_policy'] for g in groups)
    if count < 1 or total <= 0:
        raise ValueError('Each population needs positive count and total notional.')
    population_totals[population_id] = {
        'policy_count': count, 'total_notional': total,
        'mean_age_by_lives': math.fsum(g['age'] * g['count'] for g in groups) / count,
        'mean_age_by_notional': math.fsum(g['age'] * g['count'] * g['notional_per_policy']
                                          for g in groups) / total,
    }
    print(population_labels[population_id], population_totals[population_id])

left, right = (population_totals[p] for p in POPULATION_ORDER)
if left['policy_count'] != right['policy_count']:
    raise ValueError('Use the same policyholder count in both populations.')
if not math.isclose(left['total_notional'], right['total_notional'], rel_tol=1e-12):
    raise ValueError('Use the same total notional in both populations.')

populations = {
    (kind, population_id): [
        {'type': kind, 'age': g['age'], 'notional': g['notional_per_policy']}
        for g in groups for _ in range(g['count'])
    ]
    for kind in CONTRACT_TYPES for population_id, groups in population_groups.items()
}
display(cfe.Table(profiles))


**Outer simulation inputs.** One outer scenario contains one carbon/fund path,
shared by all lives, and independent mortality draws for the lives. Reusing
the same dedicated carbon and financial seeds, grid and market inputs across
all six runs reproduces the same market scenarios. This is checked below.
Mortality seeds are also held fixed, but changing ages changes death
probabilities and subsequent random-number consumption: this does **not** mean
that the two age populations have identical lifetimes.

`outer_scenarios` controls the distribution sample size; `intervals` controls
outer path discretization, hedge dates and death-settlement dates together.
`threads=16` parallelizes outer scenarios in C++; `0` uses the OpenMP runtime
limit. `progress_every=200` reports every 200 completed scenarios and the final
count; `0` disables those messages. It does not change numerical results.

The paper reports 10,000 scenarios. The editable default uses a
5,000-scenario budget and a **4,160-interval outer grid**. These are explicit
numerical choices; grid convergence should be assessed for each configuration. For a quick workflow check, temporarily use e.g. 32
scenarios and 32 intervals, with smaller initial/revaluation budgets below;
those small runs are not research estimates. Outer rows are independent;
antithetic pairs, when selected, are inside valuation only.

In [ ]:
outer_simulation = {
    'outer_scenarios': 5000,
    'intervals': 4160,
    'progress_every': 200,
    'threads': 16,
    'carbon_seed': 2026101801,
    'financial_seed': 2026101802,
    'mortality_seed': 2026101803,
}


**Initial valuation inputs.** Compute each population's premium and aggregate
delta at time zero, once, using its own ages and notionals. Do not reuse the
age-60 premium for the heterogeneous population. `paths` is the inner trajectory
budget; `intervals` is the initial pricing grid. The default uses
4,000 paths and 1,040 intervals, independently of the outer grid.

Configurations: `standard`, `standard_antithetic`, `conditioned`,
`conditioned_antithetic`. All integrate deterministic mortality; `conditioned`
also integrates the financial Gaussian analytically. Antithetic paths advance
the chosen carbon scheme with opposite Gaussian increments and work with CIR.
4,000 requested paths give 2,000 independent pair averages; odd requests round
up to a complete pair. Common seeds aid comparisons, but do not make premiums
for different ages equal. The inner routine uses the library's path sampling.

In [ ]:
initial_valuation = {
    'paths': 4000,
    'intervals': 1040,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101804,
    'financial_seed': 2026101805,
}


**Revaluation inputs.** At each intermediate outer date, value surviving
liabilities and compute their new deltas. The remaining inner grid uses

$$N(t)=\max\left(N_{\min},\left\lceil N_{\mathrm{base}}(T-t)/T\right\rceil\right).$$

`intervals` is $N_{\mathrm{base}}$, and `minimum_intervals` is $N_{\min}$,
between 1 and `intervals`. The default minimum is 4. Set both to 256 for a fixed 256-interval inner grid. This is
separate from the outer hedge grid. `paths` and `configuration` have the same
meaning as for initial valuation. All seven seed inputs across outer, initial
and revaluation streams must be distinct.

The heterogeneous portfolio has ten age cohorts rather than one. Its
mortality-weighted valuations can therefore take longer even with identical
Monte Carlo budgets and thread settings.

In [ ]:
revaluation = {
    'paths': 40,
    'intervals': 256,
    'minimum_intervals': 4,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101806,
    'financial_seed': 2026101807,
}


**Analysis inputs.** `per_notional` divides costs by total initial notional;
`aggregate` uses total currency amounts; `per_policy` divides by initial lives.
The chosen units apply to tables, combined CSVs and plots. Backend `hedging.csv`
files always retain their original aggregate present values.

Variance is sample variance (`ddof=1`), and mean SE is sample SD divided by
$\sqrt{M}$, where $M$ is the number of independent outer scenarios. Mean
intervals use a Student t approximation and are conditional on the computed
initial premium; they exclude its common Monte Carlo uncertainty and numerical
discretization error. `quantiles` and `expected_shortfall_levels` control the
distribution statistics. `comparison_family_size=18` is the Bonferroni budget
for three within-run strategy contrasts across six default experiments; it
does not adjust the cross-population bootstrap intervals below.

`bins`, `figure_formats` and `dpi` control the notebook figures. `plot_range`
sets a common x-view limit if non-null; the more specific per-contract limits
below override it. `zoom` is an interface setting for its standard strategy
plot; this notebook's Figure 6.3 has no inset and does not use that setting.
`DISPLAY_DECIMALS` affects display only; CSVs retain floating-point precision.

In [ ]:
analysis = {
    'normalization': 'per_notional',
    'confidence_level': 0.95,
    'comparison_family_size': 18,
    'quantiles': [0.025, 0.50, 0.90, 0.95, 0.975, 0.99],
    'expected_shortfall_levels': [0.95, 0.99],
    'bins': 100,
    'zoom': None,
    'plot_range': None,
    'figure_formats': ['png', 'pdf'],
    'dpi': 180,
    'title': 'Homogeneous versus heterogeneous ages',
}
DISPLAY_DECIMALS = 4


**Uncertainty inputs for the population comparison.** The reference is always
the homogeneous population. Variance reduction is
$100(1-s^2_{\mathrm{heterogeneous}}/s^2_{\mathrm{homogeneous}})$:
positive means lower heterogeneous cost variance; negative means higher.
The homogeneous row is exactly **0%**. Similar distributions, rather than a
guaranteed reduction, are the Figure 6.3 comparison.

Optional percentile bootstrap intervals resample the **same outer scenario
indices for both populations and all three strategies**, preserving their
common-market dependence. `bootstrap_repetitions=0` disables them. Their seed
is separate from simulation seeds and affects only analysis. Chunk size
controls temporary memory. These are individual intervals, conditional on the
computed initial valuations, not simultaneous confidence bands or a test
that the two full distributions are identical. Paired mean-difference
intervals use Student t approximations instead of this bootstrap.

In [ ]:
comparison_settings = {
    'bootstrap_repetitions': 2000,
    'bootstrap_seed': 2026102201,
    'bootstrap_chunk_size': 64,
}
for key in ('bootstrap_repetitions', 'bootstrap_seed', 'bootstrap_chunk_size'):
    value = comparison_settings[key]
    minimum = 1 if key == 'bootstrap_chunk_size' else 0
    if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
        raise ValueError(f'{key} must be an integer >= {minimum}.')


**Figure inputs.** The default Figure 6.3 uses dynamic costs with PE and TI on
the upper row and EI centered below. Histograms use common bin edges within
each contract, **black edges**, and the blue/orange MATLAB colors used in the
paper. Full observations enter density normalization and all statistics.
`x_limits`/`y_limits` only crop the view and never renormalize truncated data.
A constant sample is shown as a point mass. Optional `kde` uses the selected
bandwidth and grid size instead of histograms.

Change `strategy` to `static_hedge` or `no_hedge` to explore another strategy.
`make_ecdf` and `make_boxplot` toggle extra figures. `SHOW_FIGURES` and
`SAVE_FIGURES`, set in the first cell, control display and export independently.
Box-plot whiskers use a multiple of the interquartile range; outliers can be
shown or hidden. `scientific_ticks` and `scientific_power_limits` control
scientific notation for small/large values; `max_axis_ticks` limits tick
density. All styling values below can be edited.

In [ ]:
plot_settings = {
    'strategy': 'dynamic_hedge',
    'density_method': 'histogram',
    'density_grid_points': 2048,
    'kde_bandwidth': 'scott',
    'histogram_alpha': 0.60,
    'histogram_edge_color': 'black',
    'histogram_edge_width': 0.45,
    'line_width': 1.8,
    'colors': {'homogeneous': '#0072BD', 'heterogeneous': '#D95319'},
    'figure_size': [12.0, 8.0],
    'font_size': 12,
    'legend_font_size': 10,
    'grid_alpha': 0.20,
    'scientific_ticks': True,
    'scientific_power_limits': [-3, 3],
    'max_axis_ticks': 5,
    'x_limits': {'PE': None, 'TI': None, 'EI': None},
    'y_limits': {'PE': None, 'TI': None, 'EI': None},
    'make_ecdf': True,
    'make_boxplot': True,
    'boxplot_show_fliers': True,
    'boxplot_whiskers': 1.5,
}
if plot_settings['strategy'] not in cfe.STRATEGIES:
    raise ValueError('Unknown plot strategy.')
if plot_settings['density_method'] not in ('histogram', 'kde'):
    raise ValueError('density_method must be histogram or kde.')
if (not isinstance(plot_settings['density_grid_points'], int)
        or plot_settings['density_grid_points'] < 2):
    raise ValueError('Use at least two density grid points.')
if (isinstance(plot_settings['max_axis_ticks'], bool)
        or not isinstance(plot_settings['max_axis_ticks'], int)
        or plot_settings['max_axis_ticks'] < 2):
    raise ValueError('max_axis_ticks must be an integer >= 2.')
for setting in ('x_limits', 'y_limits'):
    for kind, limits in plot_settings[setting].items():
        if limits is not None and (len(limits) != 2
                or not all(math.isfinite(v) for v in limits) or limits[0] >= limits[1]):
            raise ValueError(f'{setting}/{kind} must be None or [lower, upper].')


**Build and validate the configurations.** Create one complete configuration
per contract/population pair. All market, fund, benefit, mortality and
simulation inputs match across pairs; only the individual population changes.
Each default experiment computes all three hedging strategies on the same
outcomes. Rerun this cell after changing any earlier input.

This cell validates the interface schema and displays the simulation budgets,
population counts, total notionals and mean ages before any expensive run.

In [ ]:
configs = {}
for key, people in populations.items():
    cfg = {
        'schema_version': 1,
        'experiment': 'hedging',
        'market': deepcopy(market),
        'carbon': deepcopy(carbon),
        'fund': deepcopy(fund),
        'contract': deepcopy(contract),
        'mortality': deepcopy(mortality),
        'population': deepcopy(people),
        'hedging': {
            **deepcopy(outer_simulation),
            'initial': deepcopy(initial_valuation),
            'revaluation': deepcopy(revaluation),
        },
        'analysis': deepcopy(analysis),
        'notes': ('Figure 6.3 age-composition comparison; legacy midpoint ages; '
                  'common carbon/fund streams; current time-zero cost accounting.'),
    }
    cfe.validate_config(cfg)
    configs[key] = cfg

audit = {
    'contract_types': CONTRACT_TYPES,
    'population_groups': deepcopy(population_groups),
    'population_totals': deepcopy(population_totals),
    'normalization': analysis['normalization'],
    'outer_simulation': deepcopy(outer_simulation),
    'initial_valuation': deepcopy(initial_valuation),
    'revaluation': deepcopy(revaluation),
}
print(json.dumps(audit, indent=2, allow_nan=False))
print('Experiments:', len(configs))


**Run, resume between experiments, or reload.** Each result lives in
`RUN_DIRECTORY/<contract>/<population>/`. C++ saves `hedging.csv`,
`metadata.json` and the Python launcher saves `config.json` and `console.log`.
Python generates `summary.csv`, `paired_comparisons.csv` and `analysis.json`.
The notebook also saves the exact age-cohort table and a manifest of inputs.

Reloading or reusing requires identical simulation inputs. Analysis and figure
changes are allowed; a mismatch raises an error rather than silently relabeling
old data. Reusing a completed run skips its C++ calculation. A partial run
needs a new directory or explicit overwrite. Full carbon/fund time series and
individual lifetimes are not exported; configurations and stream schedules
record how to regenerate them. Research-scale defaults have a substantial
runtime, especially for the ten-cohort population.

In [ ]:
def simulation_inputs(cfg):
    value = deepcopy({k: v for k, v in cfg.items() if k not in ('analysis', 'notes')})
    value['hedging'].setdefault('threads', 1)
    value['hedging']['revaluation'].setdefault('minimum_intervals', 1)
    return value

def write_rows(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows({k: '' if isinstance(v, (float, np.floating))
                         and not math.isfinite(v) else v for k, v in row.items()}
                        for row in rows)

results = {}
for (kind, population_id), cfg in configs.items():
    destination = RUN_DIRECTORY / kind / population_id
    completed = all((destination / name).is_file()
                    for name in ('hedging.csv', 'metadata.json'))
    reuse = not RUN_SIMULATIONS or (REUSE_COMPLETED_RUNS and completed
                                   and not OVERWRITE_EXISTING)
    print(f'\n{kind} / {population_id}: {"reload" if reuse else "run"} -> {destination}', flush=True)
    if reuse:
        result = cfe.load_result(destination)
        if simulation_inputs(result.config) != simulation_inputs(cfg):
            raise ValueError(f'{kind}/{population_id}: saved simulation inputs differ. '
                             'Use a new RUN_NAME or explicitly overwrite.')
        result.config['analysis'] = deepcopy(analysis)
        result.save_analysis(plots=False)
    else:
        result = cfe.run_experiment(cfg, destination, binary=BINARY,
            overwrite=OVERWRITE_EXISTING, plots=False, progress=PRINT_PROGRESS)
    results[(kind, population_id)] = result
    run = result.metadata['run']
    print('Initial price:', run['initial_price'])
    print('Actual outer threads:', run['threads_used'])
    print('Outer elapsed seconds:', run['simulation_wall_seconds'])

manifest = {
    **audit,
    'model_inputs': {k: deepcopy(configs[next(iter(configs))][k])
                     for k in ('market', 'carbon', 'fund', 'contract', 'mortality')},
    'analysis': deepcopy(analysis),
    'comparison_settings': deepcopy(comparison_settings),
    'plot_settings': deepcopy(plot_settings),
    'runs': [{'contract': k, 'population': p,
              'directory': str(r.directory.relative_to(RUN_DIRECTORY))}
             for (k, p), r in results.items()],
}
(RUN_DIRECTORY / 'heterogeneity_manifest.json').write_text(
    json.dumps(manifest, indent=2, allow_nan=False) + '\n', encoding='utf-8')
write_rows(RUN_DIRECTORY / 'population_profile.csv', profiles)


**Check scenario alignment, accounting and contract identities.** Every run
must have the same scenario IDs and terminal fund values. With the common
market generator, grid and dedicated seeds, this checks the shared-market
construction. The identity $C=\mathrm{claims}_{PV}-P_0-G_{PV}$ is checked for
each strategy. No hedge has zero trading gains.

Within a fixed population, PE/TI/EI use the same ordered ages and mortality
stream, so their individual deaths agree across contract types. With death
fraction $\rho$, EI claims and costs must equal PE plus $\rho$ times TI.
These are benefits on the **same lives**, not independent PE/TI portfolios.
The checks never require equal premiums, deltas or lifetimes across the two
different age populations. Initial valuations are exported for inspection.

In [ ]:
def read_raw(result):
    with (result.directory / 'hedging.csv').open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    if not rows:
        raise ValueError('Empty hedging CSV.')
    values = {key: np.array([float(row[key]) for row in rows]) for key in rows[0]}
    if not all(np.isfinite(x).all() for x in values.values()):
        raise ValueError('Raw CSV contains nonfinite values.')
    return values

def assert_close(actual, expected):
    expected = np.asarray(expected)
    scale = max(1.0, float(np.max(np.abs(expected))))
    np.testing.assert_allclose(actual, expected, rtol=1e-10, atol=1e-12 * scale)

raw = {key: read_raw(result) for key, result in results.items()}
first = next(iter(raw.values()))
expected_ids = np.arange(outer_simulation['outer_scenarios'])
initial_rows = []
for (kind, population_id), result in results.items():
    value = raw[(kind, population_id)]
    np.testing.assert_array_equal(value['scenario_id'], expected_ids)
    np.testing.assert_array_equal(value['fund_terminal'], first['fund_terminal'])
    assert result.metadata['policy_count'] == population_totals[population_id]['policy_count']
    assert_close(result.metadata['total_notional'], population_totals[population_id]['total_notional'])
    assert_close(value['initial_price'], result.metadata['run']['initial_price'])
    assert_close(value['no_hedge'], value['claims_pv'] - value['initial_price'])
    for strategy, gains in (('static_hedge', 'static_trading_gains_pv'),
                            ('dynamic_hedge', 'dynamic_trading_gains_pv')):
        assert_close(value[strategy], value['claims_pv'] - value['initial_price'] - value[gains])
    initial_rows.append({
        'contract': kind, 'population': population_id,
        'policy_count': result.metadata['policy_count'],
        'total_notional': result.metadata['total_notional'],
        'initial_price_aggregate': result.metadata['run']['initial_price'],
        'initial_price_per_notional': result.metadata['run']['initial_price'] / result.metadata['total_notional'],
        **dict(zip(('delta_PE_aggregate', 'delta_TI_aggregate', 'delta_EI_aggregate'),
                   result.metadata['run']['initial_deltas_PE_TI_EI'])),
        'threads_used': result.metadata['run']['threads_used'],
        'initial_wall_seconds': result.metadata['run']['initial_wall_seconds'],
        'outer_wall_seconds': result.metadata['run']['simulation_wall_seconds'],
    })

if set(CONTRACT_TYPES) == {'PE', 'TI', 'EI'}:
    rho = contract['endowment_death_benefit_fraction']
    for population_id in POPULATION_ORDER:
        for field in ('claims_pv', 'initial_price', *cfe.STRATEGIES):
            assert_close(raw[('EI', population_id)][field],
                raw[('PE', population_id)][field] + rho * raw[('TI', population_id)][field])
write_rows(RUN_DIRECTORY / 'initial_valuations.csv', initial_rows)
display(cfe.Table(initial_rows))
print('Shared market outcomes, PV accounting and available EI identities: verified.')


**Complete statistics.** Display and export every contract, population and
strategy: mean, variance, SD, mean SE and confidence interval, extrema,
probability of positive additional funding, selected quantiles and upper-tail
expected shortfalls. These describe realized hedging costs, rather than the
sample variance of an inner price estimator. All observations are used.

`hedging_summary.csv` combines the six per-run summaries. Also save paired
scenario data per contract for use in MATLAB or another program. These paired
CSV cost columns use the selected analysis units; the original six
`hedging.csv` files remain aggregate values.

In [ ]:
def show_rows(rows):
    if not rows:
        return
    def format_value(key, value):
        if isinstance(value, (float, np.floating)):
            if not math.isfinite(value):
                return ''
            return (f'{value:.{DISPLAY_DECIMALS}f}%' if key.endswith('_percent')
                    else f'{value:.{DISPLAY_DECIMALS}e}')
        return html.escape(str(value))
    headings = ''.join(f'<th>{html.escape(key)}</th>' for key in rows[0])
    body = ''.join('<tr>' + ''.join(f'<td>{format_value(k, v)}</td>'
                    for k, v in row.items()) + '</tr>' for row in rows)
    display(HTML('<div style="overflow:auto"><table><thead><tr>' + headings
                 + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

def divisor(result):
    return {'aggregate': 1.0, 'per_policy': result.metadata['policy_count'],
            'per_notional': result.metadata['total_notional']}[analysis['normalization']]

costs = {key: result.costs / divisor(result) for key, result in results.items()}
summaries = {key: result.summary() for key, result in results.items()}
summary_rows = [
    {'contract': kind, 'population': population_id,
     'policy_count': result.metadata['policy_count'],
     'total_notional': result.metadata['total_notional'],
     'normalization': analysis['normalization'], **row}
    for (kind, population_id), result in results.items()
    for row in summaries[(kind, population_id)]
]
show_rows(summary_rows)
write_rows(RUN_DIRECTORY / 'hedging_summary.csv', summary_rows)

for kind in CONTRACT_TYPES:
    paired_rows = []
    for j, scenario_id in enumerate(first['scenario_id']):
        paired_rows.append({
            'scenario_id': int(scenario_id), 'fund_terminal': float(first['fund_terminal'][j]),
            'normalization': analysis['normalization'],
            **{f'{p}_{s}': float(costs[(kind, p)][j, i])
               for p in POPULATION_ORDER for i, s in enumerate(cfe.STRATEGIES)},
        })
    write_rows(RUN_DIRECTORY / f'paired_costs_{kind}.csv', paired_rows)


**Compare age populations.** For each strategy, report both sample variances
and the percentage reduction from the homogeneous reference. The comparison
CSV also contains the paired mean difference (heterogeneous minus homogeneous),
its SE and individual confidence interval. Pairing refers to shared scenario
indices; it does not assume identical deaths in different age populations.

If enabled, the bootstrap exports intervals for the variance difference,
variance ratio and percentage reduction. A VR interval containing zero is
consistent with either direction at that precision; it does not establish
equivalence. Zero reference variance leaves the ratio undefined. All CSV
outputs use blank fields for undefined statistics. Do not force a favorable
variance-reduction sign: heterogeneity is not guaranteed to reduce cost risk.

In [ ]:
lookup = {(k, p, row['strategy']): row
          for (k, p), rows in summaries.items() for row in rows}
comparison_rows = []
bootstrap_rows = []
rng = np.random.default_rng(comparison_settings['bootstrap_seed'])
confidence = analysis['confidence_level']
tail = (1.0 - confidence) / 2.0

for kind in CONTRACT_TYPES:
    a = costs[(kind, 'homogeneous')]
    b = costs[(kind, 'heterogeneous')]
    n = len(a)
    for i, strategy in enumerate(cfe.STRATEGIES):
        reference = lookup[(kind, 'homogeneous', strategy)]
        for population_id in POPULATION_ORDER:
            row = lookup[(kind, population_id, strategy)]
            difference = costs[(kind, population_id)][:, i] - a[:, i]
            se = float(np.std(difference, ddof=1) / math.sqrt(n))
            mean = float(np.mean(difference))
            critical = float(student_t.isf(tail, n - 1))
            ratio = row['variance'] / reference['variance'] if reference['variance'] > 0 else math.nan
            comparison_rows.append({
                'contract': kind, 'population': population_id, 'strategy': strategy,
                'normalization': analysis['normalization'], 'n': n,
                'mean': row['mean'], 'variance': row['variance'], 'std': row['std'],
                'reference_variance': reference['variance'],
                'variance_ratio_vs_homogeneous': ratio,
                'variance_reduction_percent': 0.0 if population_id == 'homogeneous' else 100 * (1 - ratio),
                'paired_mean_difference': mean, 'paired_difference_se': se,
                'paired_difference_ci_low': mean - critical * se,
                'paired_difference_ci_high': mean + critical * se,
            })

    repetitions = comparison_settings['bootstrap_repetitions']
    if repetitions:
        sample_variances_a = np.empty((repetitions, len(cfe.STRATEGIES)))
        sample_variances_b = np.empty_like(sample_variances_a)
        chunk = comparison_settings['bootstrap_chunk_size']
        for start in range(0, repetitions, chunk):
            stop = min(start + chunk, repetitions)
            indices = rng.integers(0, n, size=(stop - start, n))
            sample_variances_a[start:stop] = np.var(a[indices], axis=1, ddof=1)
            sample_variances_b[start:stop] = np.var(b[indices], axis=1, ddof=1)
        for i, strategy in enumerate(cfe.STRATEGIES):
            va = lookup[(kind, 'homogeneous', strategy)]['variance']
            vb = lookup[(kind, 'heterogeneous', strategy)]['variance']
            delta = sample_variances_b[:, i] - sample_variances_a[:, i]
            valid = sample_variances_a[:, i] > 0
            ratios = sample_variances_b[valid, i] / sample_variances_a[valid, i]
            ratio_bounds = np.quantile(ratios, [tail, 1-tail]) if len(ratios) else [math.nan]*2
            difference_bounds = np.quantile(delta, [tail, 1-tail])
            ratio = vb / va if va > 0 else math.nan
            bootstrap_rows.append({
                'contract': kind, 'strategy': strategy,
                'normalization': analysis['normalization'], 'confidence_level': confidence,
                'bootstrap_repetitions': repetitions,
                'valid_ratio_repetitions': int(valid.sum()),
                'homogeneous_variance': va, 'heterogeneous_variance': vb,
                'variance_difference': vb - va,
                'difference_ci_low': float(difference_bounds[0]),
                'difference_ci_high': float(difference_bounds[1]),
                'variance_ratio': ratio,
                'ratio_ci_low': float(ratio_bounds[0]), 'ratio_ci_high': float(ratio_bounds[1]),
                'variance_reduction_percent': 100 * (1 - ratio),
                'VR_ci_low_percent': 100 * (1 - float(ratio_bounds[1])),
                'VR_ci_high_percent': 100 * (1 - float(ratio_bounds[0])),
            })

show_rows([r for r in comparison_rows if r['strategy'] == plot_settings['strategy']])
write_rows(RUN_DIRECTORY / 'heterogeneity_comparison.csv', comparison_rows)
bootstrap_path = RUN_DIRECTORY / 'comparison_bootstrap.csv'
if bootstrap_rows:
    show_rows([r for r in bootstrap_rows if r['strategy'] == plot_settings['strategy']])
    write_rows(bootstrap_path, bootstrap_rows)
else:
    bootstrap_path.unlink(missing_ok=True)
    print('Bootstrap disabled; no bootstrap interval CSV for this analysis.')


**Figure 6.3: cost densities.** Overlay homogeneous and heterogeneous
distributions for the selected strategy, in the paper's three-panel layout.
The default blue/orange histograms have black edges. Common bin edges span
the full data range, so each nonconstant histogram integrates to one before
any view cropping. The optional KDE is another density estimate, not an
extra simulation. All costs use the same normalization as the tables.

Figure files are saved in the run directory with the strategy in the name,
for example `figure_6_3_dynamic_hedge_density.png` and `.pdf`. The PDF is
suitable for external editing. Existing figures with the same name are
replaced when this cell is rerun; source CSVs are unchanged.

In [ ]:
contract_labels = {'PE': 'Pure Endowment', 'TI': 'Term Insurance', 'EI': 'Endowment Insurance'}
strategy_labels = dict(zip(cfe.STRATEGIES, cfe.LABELS))
units = {'aggregate': 'Aggregate cost at time zero',
         'per_policy': 'Cost per policy at time zero',
         'per_notional': 'Cost per unit total notional at time zero'}
strategy_index = cfe.STRATEGIES.index(plot_settings['strategy'])
figure_paths = []

def comparison_axes(title):
    fig = plt.figure(figsize=plot_settings['figure_size'], layout='constrained')
    if len(CONTRACT_TYPES) == 3:
        grid = fig.add_gridspec(2, 4)
        positions = {'PE': grid[0, :2], 'TI': grid[0, 2:], 'EI': grid[1, 1:3]}
    else:
        grid = fig.add_gridspec(1, len(CONTRACT_TYPES))
        positions = {kind: grid[0, i] for i, kind in enumerate(CONTRACT_TYPES)}
    axes = {kind: fig.add_subplot(positions[kind]) for kind in CONTRACT_TYPES}
    for kind, ax in axes.items():
        ax.set_title(contract_labels[kind], fontsize=plot_settings['font_size'])
        ax.tick_params(labelsize=plot_settings['font_size'])
        ax.grid(alpha=plot_settings['grid_alpha'])
        ax.set_axisbelow(True)
    fig.suptitle(title + ' | ' + strategy_labels[plot_settings['strategy']],
                 fontsize=plot_settings['font_size'] + 2)
    return fig, axes

def view_limits(kind):
    limits = plot_settings['x_limits'].get(kind)
    return analysis['plot_range'] if limits is None else limits

def full_range(values):
    lo, hi = float(values.min()), float(values.max())
    if lo == hi:
        padding = max(1.0, abs(lo)) * 1e-6
        lo, hi = lo - padding, hi + padding
    return lo, hi

def format_numeric_axes(ax, *, x=True, y=True):
    for axis in ([ax.xaxis] if x else []) + ([ax.yaxis] if y else []):
        axis.set_major_locator(MaxNLocator(nbins=plot_settings['max_axis_ticks']))
        formatter = ScalarFormatter(useMathText=True, useOffset=False)
        formatter.set_scientific(plot_settings['scientific_ticks'])
        formatter.set_powerlimits(plot_settings['scientific_power_limits'])
        axis.set_major_formatter(formatter)

def finish_figure(fig, stem):
    if SAVE_FIGURES:
        for extension in analysis['figure_formats']:
            path = RUN_DIRECTORY / f'{stem}.{extension}'
            fig.savefig(path, dpi=analysis['dpi'])
            figure_paths.append(str(path))
    if SHOW_FIGURES:
        display(fig)
    plt.close(fig)

fig, axes = comparison_axes(analysis['title'] or 'Homogeneous versus heterogeneous ages')
for kind, ax in axes.items():
    values = {p: costs[(kind, p)][:, strategy_index] for p in POPULATION_ORDER}
    lo, hi = full_range(np.concatenate(list(values.values())))
    edges = np.linspace(lo, hi, analysis['bins'] + 1)
    limits = view_limits(kind)
    grid_lo, grid_hi = (lo, hi) if limits is None else limits
    density_grid = np.linspace(grid_lo, grid_hi, plot_settings['density_grid_points'])
    for population_id, x in values.items():
        color = plot_settings['colors'][population_id]
        label = population_labels[population_id]
        if np.ptp(x) == 0:
            ax.axvline(x[0], color=color, linewidth=plot_settings['line_width'],
                       label=label + ' (point mass)')
        elif plot_settings['density_method'] == 'histogram':
            ax.hist(x, bins=edges, density=True, color=color, label=label,
                    alpha=plot_settings['histogram_alpha'],
                    edgecolor=plot_settings['histogram_edge_color'],
                    linewidth=plot_settings['histogram_edge_width'])
        else:
            kde = gaussian_kde(x, bw_method=plot_settings['kde_bandwidth'])
            ax.plot(density_grid, kde(density_grid), color=color, label=label,
                    linewidth=plot_settings['line_width'])
    if limits is not None:
        ax.set_xlim(limits)
    if plot_settings['y_limits'].get(kind) is not None:
        ax.set_ylim(plot_settings['y_limits'][kind])
    ax.set_xlabel(units[analysis['normalization']], fontsize=plot_settings['font_size'])
    ax.set_ylabel('Density', fontsize=plot_settings['font_size'])
    format_numeric_axes(ax)
    ax.legend(fontsize=plot_settings['legend_font_size'], frameon=False)
finish_figure(fig, f'figure_6_3_{plot_settings["strategy"]}_density')


**Optional distribution checks.** The ECDF uses every observation and needs
no binning or bandwidth. Box plots show medians, quartiles and whiskers, with
outliers controlled by the figure-input cell. Disable either with
`make_ecdf=False` or `make_boxplot=False`. View cropping, if selected, affects
only the ECDF view. These figures use the same raw simulation results and
analysis units; they do not launch new Monte Carlo runs.

In [ ]:
if plot_settings['make_ecdf']:
    fig, axes = comparison_axes((analysis['title'] or 'Homogeneous versus heterogeneous ages') + ': ECDF')
    for kind, ax in axes.items():
        for population_id in POPULATION_ORDER:
            x = np.sort(costs[(kind, population_id)][:, strategy_index])
            ax.step(np.r_[x[0], x], np.r_[0.0, np.arange(1, len(x) + 1) / len(x)],
                    where='post', color=plot_settings['colors'][population_id],
                    label=population_labels[population_id], linewidth=plot_settings['line_width'])
        if view_limits(kind) is not None:
            ax.set_xlim(view_limits(kind))
        ax.set_ylim(0, 1.02)
        ax.set_xlabel(units[analysis['normalization']], fontsize=plot_settings['font_size'])
        ax.set_ylabel('Cumulative probability', fontsize=plot_settings['font_size'])
        format_numeric_axes(ax)
        ax.legend(fontsize=plot_settings['legend_font_size'], frameon=False)
    finish_figure(fig, f'figure_6_3_{plot_settings["strategy"]}_ecdf')

if plot_settings['make_boxplot']:
    fig, axes = comparison_axes((analysis['title'] or 'Homogeneous versus heterogeneous ages') + ': box plots')
    for kind, ax in axes.items():
        series = [costs[(kind, p)][:, strategy_index] for p in POPULATION_ORDER]
        boxes = ax.boxplot(series, patch_artist=True,
            showfliers=plot_settings['boxplot_show_fliers'],
            whis=plot_settings['boxplot_whiskers'])
        for patch, population_id in zip(boxes['boxes'], POPULATION_ORDER):
            patch.set(facecolor=plot_settings['colors'][population_id], edgecolor='black',
                      alpha=plot_settings['histogram_alpha'])
        ax.set_xticks([1, 2], [population_labels[p] for p in POPULATION_ORDER])
        ax.set_ylabel(units[analysis['normalization']], fontsize=plot_settings['font_size'])
        format_numeric_axes(ax, x=False)
    finish_figure(fig, f'figure_6_3_{plot_settings["strategy"]}_boxplot')


**Saved files and MATLAB paths.** The files below are enough for analysis in
another program after closing the notebook. In each contract/population
directory, `hedging.csv` has scenario ID, aggregate costs for the three
strategies, discounted claims and trading gains, terminal fund and initial
premium. `metadata.json` contains count, total notional, timings, compiler/build
information, conventions and the complete configuration.

The run root holds `population_profile.csv`, `initial_valuations.csv`,
`hedging_summary.csv`, `heterogeneity_comparison.csv`, optional
`comparison_bootstrap.csv`, three `paired_costs_<contract>.csv` files, figures
and `heterogeneity_manifest.json`. Statistics use all scenarios, regardless
of display limits. No full time-series path export is implied by these files.

To use our MATLAB script for one of the contracts, copy the printed paths
into `fileMany` (homogeneous) and `fileHeterogeneous`, set `normalization` to
the same units, and disable its unrelated pooling/strategy plots if desired.
For later Python analysis, use `RUN_SIMULATIONS=False` with the same simulation
inputs and run name; the C++ executable is not needed just to reload CSVs.

In [ ]:
print('All outputs:', RUN_DIRECTORY.resolve())
for kind in CONTRACT_TYPES:
    homogeneous_csv = results[(kind, 'homogeneous')].directory / 'hedging.csv'
    heterogeneous_csv = results[(kind, 'heterogeneous')].directory / 'hedging.csv'
    print(f'\n{kind} MATLAB settings:')
    print(f'fileMany = "{homogeneous_csv}";')
    print(f'fileHeterogeneous = "{heterogeneous_csv}";')
print(f'normalization = "{analysis["normalization"]}";')
print('makePoolingPlot = false;')
print('makeStrategiesPlot = false;')

print('\nCombined CSV files:')
for path in sorted(RUN_DIRECTORY.glob('*.csv')):
    print(path.name)
print('\nFigures written in this execution:')
for path in figure_paths:
    print(path)
